# Phase 3 Colab Runner (Drive-Persistent)

This notebook runs Phase 3 scripts from a repository folder stored in Google Drive so outputs persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Configure once ----
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
REPO_BRANCH = 'master'
DRIVE_REPO_DIR = '/content/drive/MyDrive/mert-music-retrieval'

# Optional for private repos: set token via Colab Secrets or manually here
GITHUB_TOKEN = ''  # e.g. 'ghp_xxx'

In [ ]:
import os, subprocess, textwrap

os.makedirs(os.path.dirname(DRIVE_REPO_DIR), exist_ok=True)

if not os.path.exists(os.path.join(DRIVE_REPO_DIR, '.git')):
    clone_url = REPO_URL
    if GITHUB_TOKEN and REPO_URL.startswith('https://'):
        clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, DRIVE_REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', DRIVE_REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

print('Repo ready at:', DRIVE_REPO_DIR)

In [ ]:
import os

# Make scripts use Drive repo as project root
os.environ['MMR_PROJECT_ROOT'] = DRIVE_REPO_DIR
%cd {DRIVE_REPO_DIR}

!python --version
!pip -q install --upgrade pip
!pip -q install pandas scipy transformers datasets

## Block 2: Baseline Retrieval

In [ ]:
!python scripts/baseline_retrieval_run.py
!ls -lh artifacts/baseline_results.csv artifacts/baseline_rankings.csv

## Block 3: Frozen MERT Preprocess Smoke Test

In [ ]:
!python scripts/mert_frozen_run.py

## Optional: Commit/Persist Artifacts Back to GitHub

In [ ]:
# Uncomment and edit when you want to push results
# !git status
# !git add artifacts/baseline_results.csv artifacts/baseline_rankings.csv
# !git commit -m "Update baseline artifacts from Colab run"
# !git push origin {REPO_BRANCH}